# Ring Resonator Analysis, Part 3: Device-Level FSR Extraction

With hundreds of spectra uploaded across multiple dies and wafers, running FSR extraction manually would mean either a long batch script or remembering to rerun it whenever new data arrives. Here we define the analysis once as a pipeline function, and the auto-trigger handles the rest.

Each ring transmission spectrum contains a series of resonance dips. The spacing between consecutive dips is the **Free Spectral Range (FSR)**, which is related to ring geometry by:

    FSR = λ² / (ng × L)

where λ is the center wavelength, ng is the group index, and L is the ring circumference.

This notebook defines a `ring_fsr` function that detects the resonance dips in a spectrum using peak finding, computes the spacing between consecutive dips, and saves a plot and a JSON summary. We then build a pipeline that runs this analysis automatically on every uploaded spectrum.

## Setup

In [ ]:
import getpass
import json
from pathlib import Path

import gfhub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gfhub import nodes
from PIL import Image
from scipy.signal import find_peaks
from tqdm.auto import tqdm

client = gfhub.Client()
user = getpass.getuser()
print(f"Running as user: {user}")

## Load a sample spectrum

Before building the pipeline, we load one spectrum interactively to understand the data structure and check that the peak finder works correctly.

In [ ]:
upload_id = client.query_files(
    name="ring_5000.parquet", tags=["project:tutorial_rings", user, "die:0,0"]
).newest()["id"]

sample_path = Path("sample_df.parquet").resolve()
sample_df = pd.read_parquet(client.download_file(upload_id, sample_path))
print(sample_df.head())

## Defining the FSR analysis function

`ring_fsr` uses `scipy.signal.find_peaks` to detect minima in the transmission (by finding peaks in the negated signal). The `prominence` parameter controls the minimum depth that a dip must have to count as a resonance, which helps reject noise. If fewer than two resonances are found, the function records `None` for all FSR quantities rather than crashing.

The JSON output includes `fsr_mean` and `fsr_std` in µm, which the die-level pipeline in notebook 4 will read and convert to nm.

In [ ]:
def ring_fsr(
    path: Path,
    /,
    *,
    xname: str = "wl",
    yname: str = "power",
    peaks_prominence: float = 0.01,
) -> tuple[Path, Path]:
    """Detect resonance dips in a ring transmission spectrum and compute the FSR.

    Args:
        path: Path to Parquet file with columns `xname` and `yname`.
        xname: Column name for wavelength.
        yname: Column name for transmitted power.
        peaks_prominence: Minimum prominence for a dip to be counted as a resonance.

    Returns:
        Tuple of (plot PNG path, results JSON path).
    """
    df = pd.read_parquet(path)
    wavelength = df[xname].values
    power = df[yname].values

    # Resonances are minima in transmission; find_peaks works on maxima, so negate
    peaks, _ = find_peaks(-power, prominence=peaks_prominence)

    if len(peaks) < 2:
        fsr_mean, fsr_std, fsr_values = None, None, []
    else:
        peak_wavelengths = wavelength[peaks]
        fsr_values = np.diff(peak_wavelengths).tolist()
        fsr_mean = float(np.mean(fsr_values))
        fsr_std = float(np.std(fsr_values))

    fig, ax = plt.subplots()
    ax.plot(wavelength, power, color="C0", label="Spectrum")
    if len(peaks) > 0:
        ax.scatter(wavelength[peaks], power[peaks], s=50, zorder=5, color="C1", marker="x", label="Resonances")
    ax.set_xlabel("Wavelength (µm)")
    ax.set_ylabel("Power")
    title = f"FSR = {fsr_mean:.4f} µm" if fsr_mean is not None else "FSR: insufficient peaks"
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plot_path = path.with_name(path.stem + "_fsr_analysis.png")
    plt.savefig(plot_path, bbox_inches="tight", dpi=100)
    plt.close()

    results = {
        "fsr_mean": fsr_mean,
        "fsr_std": fsr_std,
        "num_peaks": len(peaks),
        "peak_wavelengths": [float(w) for w in wavelength[peaks]],
    }
    results_path = path.with_name(path.stem + "_fsr_analysis.json")
    results_path.write_text(json.dumps(results, indent=2))

    return plot_path, results_path

In [ ]:
func_def = gfhub.Function(
    func=ring_fsr,
    dependencies={
        "json": "import json",
        "numpy": "import numpy as np",
        "scipy": "from scipy.signal import find_peaks",
        "pandas[pyarrow]": "import pandas as pd",
        "matplotlib": "import matplotlib.pyplot as plt",
    },
)

## Testing locally

For a tuple-return function, `result["output"]` is a list with one element per return value.

In [ ]:
result = func_def.eval(sample_path)
plot_path, json_path = result["output"]
print(json_path.read_text())
Image.open(plot_path)

In [ ]:
client.add_function(func_def)
print("ring_fsr uploaded.")

## Creating the FSR analysis pipeline

This pipeline has both an auto-trigger (fires on every new spectrum upload) and a manual trigger (used below to backfill existing files). The auto-trigger means peak finding runs on every new spectrum as it is uploaded, so the FSR results stay current without a manual batch step. Because the function produces two outputs (plot and JSON), we use two separate `save` nodes, each tagged with the original file's provenance.

In [ ]:
p = gfhub.Pipeline()

p.auto_trigger = nodes.on_file_upload(tags=[".parquet", "project:tutorial_rings", user])
p.manual_trigger = nodes.on_manual_trigger()
p.load_file = nodes.load()
p.load_tags = nodes.load_tags()

p += p.auto_trigger >> p.load_file
p += p.manual_trigger >> p.load_file
p += p.auto_trigger >> p.load_tags
p += p.manual_trigger >> p.load_tags

p.fsr_analysis = nodes.function(
    function="ring_fsr",
    kwargs={"xname": "wl", "yname": "power", "peaks_prominence": 0.01},
)
p += p.load_file >> p.fsr_analysis

p.save_plot = nodes.save()
p.save_json = nodes.save()
p += p.fsr_analysis[0] >> p.save_plot[0]
p += p.load_tags >> p.save_plot[1]
p += p.fsr_analysis[1] >> p.save_json[0]
p += p.load_tags >> p.save_json[1]

confirmation = client.add_pipeline(name="rings_fsr_analysis", schema=p)
print(f"Pipeline ready: {client.pipeline_url(confirmation['id'])}")

## Run analysis on all existing files

In [ ]:
device_files = client.query_files(tags=["project:tutorial_rings", user, ".parquet", "device"])
print(f"Found {len(device_files)} device files")

job_ids = []
for device_file in tqdm(device_files):
    triggered = client.trigger_pipeline("rings_fsr_analysis", device_file["id"])
    job_ids.extend(triggered["job_ids"])

print(f"Triggered {len(job_ids)} analysis jobs")

In [ ]:
jobs = client.wait_for_jobs(job_ids)
print(f"All jobs complete. Statuses: {set(j['status'] for j in jobs)}")

## View a sample result

In [ ]:
analysis_plots = client.query_files(name="*_fsr_analysis.png", tags=["project:tutorial_rings", user])
print(f"Found {len(analysis_plots)} analysis plots")

if analysis_plots:
    img = Image.open(client.download_file(analysis_plots[0]["id"]))
    display(img.resize((530, 400)))

In [ ]:
analysis_results = client.query_files(name="*_fsr_analysis.json", tags=["project:tutorial_rings", user])
print(f"Found {len(analysis_results)} analysis result files")

if analysis_results:
    result_data = json.load(client.download_file(analysis_results[0]["id"]))
    print(json.dumps(result_data, indent=2))

## What's next?

Each device now has an FSR measurement stored in DataLab. The next notebook groups these by die and by ring radius, computes the mean FSR per (die, radius) group, and saves the results for wafer-level aggregation.